In [3]:
import os
import base64
import time
import subprocess
from IPython.display import HTML, Audio, display, Javascript
from google.colab import drive
from google.colab.output import eval_js

# ==========================================
# 1. MOUNT GOOGLE DRIVE & SETUP FOLDERS
# ==========================================
print("🔗 Mounting Google Drive...")
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/Rafeeq_rec'   ## first create folder in your drive (Rafeeq_rec)
COMMANDS = [
    "move_forward", "move_backward", "turn_right", "turn_left",
    "stop", "sleep", "rafeeq", "go_to_bathroom",
    "go_to_bedroom", "go_to_livingroom", "go_to_kitchen"
]

print("\n📁 Checking/Creating Folders...")
for cmd in COMMANDS:
    folder_path = os.path.join(BASE_DIR, cmd)
    os.makedirs(folder_path, exist_ok=True)
print(f"✅ All 11 folders are ready inside: {BASE_DIR}")

# ==========================================
# 2. JAVASCRIPT RECORDER (Browser Microphone)
# ==========================================
# This JS creates a UI button to record audio from your browser
RECORD_JS = """
async function recordAudio() {
  const div = document.createElement('div');
  const startButton = document.createElement('button');
  const stopButton = document.createElement('button');

  startButton.textContent = '🎙️ START RECORDING';
  startButton.style.cssText = 'font-size: 20px; padding: 15px; background-color: #4CAF50; color: white; border: none; border-radius: 5px; cursor: pointer;';

  stopButton.textContent = '🛑 STOP RECORDING';
  stopButton.style.cssText = 'font-size: 20px; padding: 15px; background-color: #f44336; color: white; border: none; border-radius: 5px; cursor: pointer; display: none;';

  document.body.appendChild(div);
  div.appendChild(startButton);
  div.appendChild(stopButton);

  const stream = await navigator.mediaDevices.getUserMedia({audio:true});
  const mediaRecorder = new MediaRecorder(stream);
  const chunks = [];

  mediaRecorder.ondataavailable = e => chunks.push(e.data);

  return new Promise(resolve => {
    startButton.onclick = () => {
      mediaRecorder.start();
      startButton.style.display = 'none';
      stopButton.style.display = 'inline-block';
    };

    stopButton.onclick = () => {
      mediaRecorder.stop();
      stream.getTracks().forEach(track => track.stop());
    };

    mediaRecorder.onstop = () => {
      const blob = new Blob(chunks);
      const reader = new FileReader();
      reader.readAsDataURL(blob);
      reader.onloadend = () => {
        resolve(reader.result);
        div.remove();
      };
    };
  });
}
"""

def record_audio_to_file(filepath):
    """Records audio from browser and saves it as a strict 16kHz WAV file."""
    display(Javascript(RECORD_JS))
    # Wait for the user to click start and stop, then get the base64 audio string
    s = eval_js('recordAudio()')

    # Strip the base64 header
    b64 = s.split(',')[1]

    # Save the raw browser audio (usually WebM format) to a temporary file
    temp_file = "temp_audio.webm"
    with open(temp_file, "wb") as f:
        f.write(base64.b64decode(b64))

    # Convert WebM to 16kHz, 16-bit Mono WAV using FFmpeg (Required for ML Training)
    # -y (overwrite), -i (input), -ac 1 (mono), -ar 16000 (16kHz sample rate)
    subprocess.run([
        'ffmpeg', '-y', '-i', temp_file,
        '-ac', '1', '-ar', '16000', filepath
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    os.remove(temp_file)

# ==========================================
# 3. MAIN RECORDING LOOP
# ==========================================
print("\n" + "="*50)
print(" 🎙️ RAFEEQ DATASET COLLECTOR")
print("="*50)

try:
    n_samples = int(input("How many times do you want to record EACH command? (e.g., 5 or 10): "))
except ValueError:
    print("❌ Invalid number. Defaulting to 5.")
    n_samples = 5

for cmd in COMMANDS:
    print(f"\n" + "="*40)
    print(f" 📢 COMMAND: '{cmd.upper().replace('_', ' ')}'")
    print("="*40)

    for i in range(1, n_samples + 1):
        filename = f"{cmd}_{i}.wav"
        save_path = os.path.join(BASE_DIR, cmd, filename)

        # Check if file already exists so you can pause and resume later
        if os.path.exists(save_path):
            print(f"⏩ Skipping {filename} (Already exists)")
            continue

        print(f"\nRecording {i}/{n_samples} for '{cmd}'...")
        print("Click START, speak clearly, then click STOP.")

        # Trigger the recording interface
        record_audio_to_file(save_path)

        print(f"✅ Saved to Drive: {cmd}/{filename}")
        time.sleep(0.5) # Short pause between recordings

print("\n🎉 DATASET COLLECTION COMPLETE! All files are in your Google Drive.")

🔗 Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📁 Checking/Creating Folders...
✅ All 11 folders are ready inside: /content/drive/MyDrive/Rafeeq_rec

 🎙️ RAFEEQ DATASET COLLECTOR
How many times do you want to record EACH command? (e.g., 5 or 10): 10

 📢 COMMAND: 'MOVE FORWARD'
⏩ Skipping move_forward_1.wav (Already exists)
⏩ Skipping move_forward_2.wav (Already exists)
⏩ Skipping move_forward_3.wav (Already exists)
⏩ Skipping move_forward_4.wav (Already exists)
⏩ Skipping move_forward_5.wav (Already exists)

Recording 6/10 for 'move_forward'...
Click START, speak clearly, then click STOP.


<IPython.core.display.Javascript object>

✅ Saved to Drive: move_forward/move_forward_6.wav

Recording 7/10 for 'move_forward'...
Click START, speak clearly, then click STOP.


<IPython.core.display.Javascript object>

✅ Saved to Drive: move_forward/move_forward_7.wav

Recording 8/10 for 'move_forward'...
Click START, speak clearly, then click STOP.


...

✅ Saved to Drive: go_to_kitchen/go_to_kitchen_8.wav

Recording 9/10 for 'go_to_kitchen'...
Click START, speak clearly, then click STOP.


<IPython.core.display.Javascript object>

✅ Saved to Drive: go_to_kitchen/go_to_kitchen_9.wav

Recording 10/10 for 'go_to_kitchen'...
Click START, speak clearly, then click STOP.


<IPython.core.display.Javascript object>

✅ Saved to Drive: go_to_kitchen/go_to_kitchen_10.wav

🎉 DATASET COLLECTION COMPLETE! All files are in your Google Drive.


In [4]:
import os
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import layers, models

# ==========================================
# 1. CONFIGURATION & SETUP
# ==========================================
BASE_DIR = '/content/drive/MyDrive/Rafeeq_rec'
SAMPLE_RATE = 16000
DURATION = 1.5  # Standardize all clips to 1.5 seconds
SAMPLES_PER_TRACK = int(SAMPLE_RATE * DURATION)

print("🔗 Connecting to Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

# ==========================================
# 2. FEATURE EXTRACTION (MFCCs)
# ==========================================
def extract_features(file_path):
    """Loads audio, standardizes length, and extracts MFCC features."""
    # Load audio
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE)

    # Trim dead silence from the edges
    y, _ = librosa.effects.trim(y, top_db=20)

    # Standardize length: Pad with silence if too short, cut if too long
    if len(y) > SAMPLES_PER_TRACK:
        y = y[:SAMPLES_PER_TRACK]
    else:
        padding = SAMPLES_PER_TRACK - len(y)
        y = np.pad(y, (0, padding), 'constant')

    # Extract MFCCs (creates a 2D array representing the sound wave)
    mfccs = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=13, n_fft=2048, hop_length=512)
    return mfccs

print("\n⚙️ Extracting audio features... (This might take a minute)")
X = []
y_labels = []

# Loop through all 11 folders
for command in os.listdir(BASE_DIR):
    folder_path = os.path.join(BASE_DIR, command)
    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            if file.endswith('.wav'):
                file_path = os.path.join(folder_path, file)
                try:
                    features = extract_features(file_path)
                    X.append(features)
                    y_labels.append(command)
                except Exception as e:
                    print(f"Error processing {file}: {e}")

X = np.array(X)
y_labels = np.array(y_labels)

# Reshape X for the CNN (Adding a channel dimension like an image)
X = X[..., np.newaxis]
print(f"✅ Extracted features for {len(X)} audio files. Shape: {X.shape}")

# ==========================================
# 3. PREPARE DATA FOR TRAINING
# ==========================================
# Convert text labels ("move_forward") to numbers (0, 1, 2...)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_labels)
num_classes = len(label_encoder.classes_)

# Print the label mapping so you know which number is which command
print("\n🏷️ Label Mapping:")
for i, name in enumerate(label_encoder.classes_):
    print(f"   {i} -> {name}")

# Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# ==========================================
# 4. BUILD THE TINY NEURAL NETWORK
# ==========================================
print("\n🧠 Building the Neural Network...")
input_shape = (X.shape[1], X.shape[2], 1)

model = models.Sequential([
    layers.Conv2D(16, (3, 3), activation='relu', input_shape=input_shape),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2), # Dropout prevents the model from just memorizing the data

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax') # Softmax gives probabilities for the 11 classes
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# ==========================================
# 5. TRAIN THE MODEL
# ==========================================
print("\n🚀 Training started...")
# Training for 30 epochs (passes over the data)
history = model.fit(X_train, y_train, epochs=50, batch_size=8, validation_data=(X_test, y_test))

# ==========================================
# 6. CONVERT TO TENSORFLOW LITE (For Raspberry Pi)
# ==========================================
print("\n📦 Converting model to TensorFlow Lite format for Raspberry Pi...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the model to your Drive
tflite_path = '/content/drive/MyDrive/Rafeeq_rec/rafeeq_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

# Save the labels so the Pi knows the names of the commands
labels_path = '/content/drive/MyDrive/Rafeeq_rec/labels.txt'
with open(labels_path, 'w') as f:
    for label in label_encoder.classes_:
        f.write(f"{label}\n")

print("\n🎉 SUCCESS! Your custom AI is trained.")
print(f"💾 Model saved to: {tflite_path}")
print(f"📝 Labels saved to: {labels_path}")

🔗 Connecting to Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

⚙️ Extracting audio features... (This might take a minute)
✅ Extracted features for 110 audio files. Shape: (110, 13, 47, 1)

🏷️ Label Mapping:
   0 -> go_to_bathroom
   1 -> go_to_bedroom
   2 -> go_to_kitchen
   3 -> go_to_livingroom
   4 -> move_backward
   5 -> move_forward
   6 -> rafeeq
   7 -> sleep
   8 -> stop
   9 -> turn_left
   10 -> turn_right

🧠 Building the Neural Network...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 11, 45, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 5, 22, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 5, 22, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 3, 20, 32)      │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 1, 10, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1, 10, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 320)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 11)             │           715 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,059 (101.79 KB)

 Trainable params: 26,059 (101.79 KB)

 Non-trainable params: 0 (0.00 B)


🚀 Training started...
Epoch 1/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.1242 - loss: 36.5034 - val_accuracy: 0.0909 - val_loss: 8.8677
Epoch 2/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1181 - loss: 16.2125 - val_accuracy: 0.1818 - val_loss: 9.5787
Epoch 3/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1817 - loss: 15.7333 - val_accuracy: 0.0909 - val_loss: 3.5345
Epoch 4/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1623 - loss: 8.7847 - val_accuracy: 0.0455 - val_loss: 3.0105
Epoch 5/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1658 - loss: 5.8536 - val_accuracy: 0.0909 - val_loss: 2.3166
Epoch 6/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2467 - loss: 4.3046 - val_accuracy: 0.3182 - val_loss: 2.0975
Epoch 7/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3073 - loss: 3.4261 - val_accuracy: 0.3636 - val_loss: 1.7414
Epoch 8/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2387 - loss: 3.1379 